# Clase 8: Transfer Learning y Fine-Tuning

## Ejemplos prácticos para la clase

---
## 1. El problema: entrenar desde cero es carísimo

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# ¿Cuánto cuesta entrenar modelos famosos?
modelos_info = [
    ("ResNet-50", 25, "ImageNet (14M imgs)", "1 semana, 8 GPUs"),
    ("EfficientNet-B7", 66, "ImageNet (14M imgs)", "~4 días, TPU"),
    ("ViT-Large", 307, "ImageNet-21K (14M imgs)", "~30 días, TPU Pod"),
    ("GPT-3", 175000, "Text (300B tokens)", "~$4.6M USD"),
]

print("=" * 70)
print("Costo de entrenar modelos desde cero")
print("=" * 70)
print(f"{'Modelo':<20} {'Params (M)':>12} {'Datos':>22} {'Costo':>20}")
print("-" * 75)
for nombre, params, datos, costo in modelos_info:
    print(f"{nombre:<20} {params:>12,} {datos:>22} {costo:>20}")

print(f"\nVos tenés: 2000 imágenes y una laptop. ¿Qué hacés?")
print(f"→ TRANSFER LEARNING: usás un modelo que ya sabe ver.")

---
## 2. Transfer Learning: reusar lo aprendido

In [ ]:
# Cargar ResNet-50 pre-entrenada en ImageNet

model = models.resnet50(weights='IMAGENET1K_V2')

# Ver la estructura
print("Estructura de ResNet-50 (últimas capas):")
children = list(model.named_children())
for name, module in children[-3:]:
    print(f"  {name}: {module.__class__.__name__}")

print(f"\nÚltima capa (fc): {model.fc}")
print(f"  → Sale con 1000 clases (las de ImageNet)")

total = sum(p.numel() for p in model.parameters())
print(f"\nParámetros totales: {total:,}")

In [ ]:
# Transfer Learning: congelar todo, reemplazar la última capa

# 1. Congelar TODAS las capas
for param in model.parameters():
    param.requires_grad = False

# 2. Reemplazar la última capa (1000 clases → 2 clases: perro/gato)
num_features = model.fc.in_features  # 2048
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2)  # 2 clases: perro y gato
)

# Contar parámetros entrenables vs totales
params_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
params_totales = sum(p.numel() for p in model.parameters())
params_congelados = params_totales - params_entrenables

print("=" * 50)
print("Transfer Learning: ResNet-50 → Perro/Gato")
print("=" * 50)
print(f"  Parámetros congelados:  {params_congelados:>12,} ({params_congelados/params_totales:.1%})")
print(f"  Parámetros entrenables: {params_entrenables:>12,} ({params_entrenables/params_totales:.1%})")
print(f"  Parámetros totales:     {params_totales:>12,}")
print(f"\n¡Solo entrenamos el {params_entrenables/params_totales:.1%} de los parámetros!")
print(f"El 97.8% ya sabe detectar bordes, texturas, formas, etc.")

In [ ]:
# Visualizar qué se congela y qué se entrena

capas = []
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    if n_params > 0:
        capas.append((name, n_params, trainable))

fig, ax = plt.subplots(figsize=(12, 5))
nombres = [c[0] for c in capas]
total_p = [c[1] for c in capas]
train_p = [c[2] for c in capas]

bars = ax.barh(range(len(capas)), total_p, color='lightcoral', label='Congelados')
ax.barh(range(len(capas)), train_p, color='#2ecc71', label='Entrenables')
ax.set_yticks(range(len(capas)))
ax.set_yticklabels(nombres)
ax.set_xlabel('Cantidad de parámetros')
ax.set_title('Transfer Learning: qué se congela y qué se entrena')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 3. Demo: Transfer Learning con CIFAR-10

In [ ]:
# Preparar datos (CIFAR-10 como ejemplo, con las transforms de ImageNet)

transform_train = transforms.Compose([
    transforms.Resize(224),                # ResNet espera 224x224
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Usamos un subconjunto para que sea rápido (simula tener pocos datos)
full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
full_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Tomar solo 500 muestras de entrenamiento (simular dataset chico)
train_subset = torch.utils.data.Subset(full_train, range(500))
test_subset = torch.utils.data.Subset(full_test, range(200))

train_loader = torch.utils.data.DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_subset, batch_size=32)

print(f"Entrenamiento: {len(train_subset)} imágenes (simulando dataset chico)")
print(f"Test: {len(test_subset)} imágenes")

In [ ]:
# Comparar: Entrenar desde cero vs Transfer Learning

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

def entrenar_y_evaluar(model, train_loader, test_loader, num_epochs, lr, nombre):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    # Solo entrenar parámetros con requires_grad=True
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    
    train_accs = []
    for epoch in range(num_epochs):
        model.train()
        correct, total = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            _, predicted = model(images).max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        train_accs.append(100. * correct / total)
    
    # Evaluar
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = model(images).max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    test_acc = 100. * correct / total
    
    print(f"{nombre}: Train Acc = {train_accs[-1]:.1f}%, Test Acc = {test_acc:.1f}%")
    return train_accs, test_acc

In [ ]:
# Modelo A: CNN entrenada desde cero
print("Entrenando CNN desde cero...")
cnn_scratch = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.AdaptiveAvgPool2d((4, 4)),
    nn.Flatten(),
    nn.Linear(64 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 10)
)
accs_scratch, test_scratch = entrenar_y_evaluar(
    cnn_scratch, train_loader, test_loader, num_epochs=10, lr=0.001, nombre="Desde cero"
)

# Modelo B: ResNet-50 con Transfer Learning
print("\nTransfer Learning con ResNet-50...")
resnet_tl = models.resnet50(weights='IMAGENET1K_V2')
for param in resnet_tl.parameters():
    param.requires_grad = False
resnet_tl.fc = nn.Sequential(
    nn.Linear(resnet_tl.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10)
)
accs_tl, test_tl = entrenar_y_evaluar(
    resnet_tl, train_loader, test_loader, num_epochs=10, lr=0.001, nombre="Transfer Learning"
)

In [ ]:
# Visualizar la comparación

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, 11), accs_scratch, 'o-', color='#e74c3c', linewidth=2, label='Desde cero')
axes[0].plot(range(1, 11), accs_tl, 'o-', color='#2ecc71', linewidth=2, label='Transfer Learning')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Accuracy (%)')
axes[0].set_title('Aprendizaje durante el entrenamiento')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

bars = axes[1].bar(['Desde cero', 'Transfer\nLearning'], [test_scratch, test_tl],
                    color=['#e74c3c', '#2ecc71'], width=0.5)
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Accuracy en Test')
for bar, val in zip(bars, [test_scratch, test_tl]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Transfer Learning vs Entrenar desde cero ({len(train_subset)} imágenes)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nCon solo {len(train_subset)} imágenes:")
print(f"  Desde cero:       {test_scratch:.1f}%")
print(f"  Transfer Learning: {test_tl:.1f}%")
print(f"  Diferencia: +{test_tl - test_scratch:.1f} puntos porcentuales")

---
## 4. Fine-Tuning: ajuste fino

In [ ]:
# Fine-Tuning: descongelar las últimas capas y re-entrenar con lr bajo

# Partimos del modelo de Transfer Learning
model_ft = models.resnet50(weights='IMAGENET1K_V2')

# Congelar todo primero
for param in model_ft.parameters():
    param.requires_grad = False

# Reemplazar última capa
model_ft.fc = nn.Sequential(
    nn.Linear(model_ft.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10)
)

# Descongelar layer4 (última capa convolucional)
for param in model_ft.layer4.parameters():
    param.requires_grad = True

# Contar parámetros
params_train = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
params_total = sum(p.numel() for p in model_ft.parameters())

print("Fine-Tuning: descongelamos layer4 + fc")
print(f"  Entrenables: {params_train:,} ({params_train/params_total:.1%})")
print(f"  Totales:     {params_total:,}")

# Optimizer con learning rates diferenciados
optimizer_ft = torch.optim.Adam([
    {'params': model_ft.layer4.parameters(), 'lr': 1e-5},   # lr bajo para capas pre-entrenadas
    {'params': model_ft.fc.parameters(), 'lr': 1e-3},        # lr normal para capa nueva
])

print(f"\nDiscriminative Learning Rates:")
print(f"  layer4 (pre-entrenada): lr = 1e-5 (ajustes mínimos)")
print(f"  fc (nueva):            lr = 1e-3 (aprende rápido)")

---
## 5. ¿Cuándo usar cada estrategia?

In [ ]:
# Matriz de decisión visual

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Cuadrantes
ax.fill_between([0, 5], [5, 5], [10, 10], color='#3498db', alpha=0.15)
ax.fill_between([5, 10], [5, 5], [10, 10], color='#2ecc71', alpha=0.15)
ax.fill_between([0, 5], [0, 0], [5, 5], color='#f39c12', alpha=0.15)
ax.fill_between([5, 10], [0, 0], [5, 5], color='#e74c3c', alpha=0.15)

# Texto en cada cuadrante
ax.text(2.5, 7.5, 'Transfer Learning\n(congelar todo)', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#2c3e50')
ax.text(2.5, 6.5, 'Ej: razas de perros\ncon 1000 imágenes', ha='center', va='center',
        fontsize=10, color='gray')

ax.text(7.5, 7.5, 'Fine-Tuning\n(descongelar últimas capas)', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#2c3e50')
ax.text(7.5, 6.5, 'Ej: productos e-commerce\ncon 100K imágenes', ha='center', va='center',
        fontsize=10, color='gray')

ax.text(2.5, 2.5, 'TL + features\nintermedias', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#2c3e50')
ax.text(2.5, 1.5, 'Ej: rayos X\ncon 500 imágenes', ha='center', va='center',
        fontsize=10, color='gray')

ax.text(7.5, 2.5, 'Fine-Tuning agresivo\no desde cero', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#2c3e50')
ax.text(7.5, 1.5, 'Ej: imágenes satelitales\ncon millones de imgs', ha='center', va='center',
        fontsize=10, color='gray')

# Ejes
ax.axhline(y=5, color='black', linewidth=1)
ax.axvline(x=5, color='black', linewidth=1)
ax.set_xlabel('Cantidad de datos →', fontsize=13)
ax.set_ylabel('Similitud con ImageNet →', fontsize=13)
ax.set_xticks([2.5, 7.5])
ax.set_xticklabels(['Pocos datos', 'Muchos datos'], fontsize=11)
ax.set_yticks([2.5, 7.5])
ax.set_yticklabels(['Tarea diferente', 'Tarea similar'], fontsize=11)
ax.set_title('¿Cuándo usar cada estrategia?', fontsize=15)

plt.tight_layout()
plt.show()

---
## 6. Hugging Face: el supermercado de modelos

In [ ]:
# Usar modelos de Hugging Face con timm (visión)

try:
    import timm
    
    # Listar algunos modelos disponibles
    modelos_populares = ['resnet50', 'efficientnet_b0', 'vit_base_patch16_224', 'convnext_base']
    
    print("Modelos disponibles en timm:")
    print(f"  Total: {len(timm.list_models()):,} modelos")
    print(f"  ResNets: {len(timm.list_models('resnet*'))}")
    print(f"  EfficientNets: {len(timm.list_models('efficientnet*'))}")
    print(f"  Vision Transformers: {len(timm.list_models('vit*'))}")
    
    # Crear un modelo pre-entrenado con timm
    print(f"\nCrear modelo con timm es una línea:")
    print(f'  model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2)')
    
    # Demo
    model_timm = timm.create_model('efficientnet_b0', pretrained=True, num_classes=10)
    params = sum(p.numel() for p in model_timm.parameters())
    print(f"\nEfficientNet-B0: {params:,} parámetros")
    
    # Test
    dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = model_timm(dummy)
    print(f"Output shape: {out.shape}")
    
except ImportError:
    print("timm no está instalado. Instalalo con: pip install timm")

In [ ]:
# Hugging Face Transformers para NLP

try:
    from transformers import pipeline
    
    # Análisis de sentimiento con un modelo pre-entrenado
    print("Transfer Learning en NLP con Hugging Face:")
    print("  Cargando modelo de análisis de sentimiento...")
    
    clasificador = pipeline("sentiment-analysis")
    
    textos = [
        "Me encanta la programación, es lo mejor del mundo",
        "This movie was terrible and boring",
        "The weather is nice today",
        "I hate when my code has bugs",
    ]
    
    print(f"\n{'Texto':<50} {'Sentimiento':>12} {'Confianza':>10}")
    print("-" * 75)
    for texto in textos:
        result = clasificador(texto)[0]
        print(f"{texto:<50} {result['label']:>12} {result['score']:>10.2%}")
    
    print(f"\nEste modelo fue pre-entrenado en millones de textos.")
    print(f"Funciona 'out of the box' sin entrenar nada.")
    
except ImportError:
    print("transformers no está instalado. Instalalo con: pip install transformers")

---
## 7. Workflow práctico completo

In [ ]:
# Resumen del workflow completo

print("""
╔══════════════════════════════════════════════════════════════╗
║              WORKFLOW DE TRANSFER LEARNING                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  1. ELEGIR MODELO PRE-ENTRENADO                            ║
║     - ResNet-50: buen balance general                      ║
║     - EfficientNet-B0: recursos limitados                  ║
║     - ViT-Base: si tenés muchos datos                      ║
║                                                            ║
║  2. PREPARAR DATOS                                         ║
║     - Mismas transforms que el modelo original             ║
║     - Resize a 224x224                                     ║
║     - Normalizar con mean/std de ImageNet                  ║
║     - Data augmentation para train                         ║
║                                                            ║
║  3. FASE 1: TRANSFER LEARNING                              ║
║     - Congelar backbone                                    ║
║     - Entrenar solo clasificador                           ║
║     - lr = 1e-3, pocos epochs (5-10)                       ║
║                                                            ║
║  4. FASE 2: FINE-TUNING                                    ║
║     - Descongelar últimas capas                            ║
║     - lr MUY BAJO para capas pre-entrenadas (1e-5)         ║
║     - lr normal para capa nueva (1e-3)                     ║
║     - Más epochs (10-20)                                   ║
║                                                            ║
║  5. EVALUAR Y GUARDAR                                      ║
║     - torch.save(model.state_dict(), 'modelo.pth')         ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## Resumen

| Estrategia | Cuándo | Qué hacés |
|-----------|--------|----------|
| Desde cero | Muchos datos + tarea muy diferente | Diseñar y entrenar todo |
| Transfer Learning | Pocos datos | Congelar todo, entrenar solo la cabeza |
| Fine-Tuning | Datos medianos/muchos | Descongelar últimas capas, lr bajo |

| Herramienta | Para qué |
|------------|----------|
| `torchvision.models` | Modelos pre-entrenados de PyTorch |
| `timm` | +1000 modelos de visión |
| `transformers` | Modelos de NLP (BERT, GPT, etc.) |
| Hugging Face Hub | Supermercado de +500K modelos |